### Sudokopie – Lösung Schritt für Schritt

*41. Bundeswettbewerb Informatik (2022), 1. Runde, Aufgabe 3*

Rudi ist gar nicht mehr begeistert von den Sudokus seiner Rätselzeitung: Er hat den starken Verdacht, dass die Redaktion die
Sudoku-Aufgaben immer wiederverwendet! Die Rätsel sehen zwar unterschiedlich aus, aber der Lösungsweg wiederholt sich erkennbar.
Er möchte nun seinen Verdacht nachweisen.

Rudi überlegt sich, dass man aus einem Sudoku sehr viele Varianten erzeugen kann, indem man eine oder mehrere der folgenden
Umformungen durchführt:

- Permutation (Umordnung) der drei Spalten innerhalb der Spaltenblöcke.
- Permutation der drei Spaltenblöcke.
- Permutation der drei Zeilen innerhalb der Zeilenblöcke.
- Permutation der drei Zeilenblöcke.
- 90-Grad-Rotation im Uhrzeigersinn.
- Umbenennen der Ziffern 1 bis 9 (z. B. alle ‚8‘en und ‚3‘en vertauschen).

Zwei Rätsel auf dem Aufgabenblatt sind Varianten voneinander: Dort wurden der 1. und der 3. Spaltenblock sowie die 5. und die
6. Zeile vertauscht und die Ziffern umbenannt: 1 → 2, 2 → 3, …, 8 → 9, 9 → 1.

#### Aufgabe 3

Hilf Rudi, indem du ein Programm schreibst, das zwei Sudokus einliest und überprüft, ob sie Varianten voneinander sind. Wenn ja,
soll das Programm Umformungen ausgeben, mit denen das eine aus dem anderen Sudoku erzeugt werden kann.

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest, und dokumentiere die Ergebnisse.

#### Beispieldaten (`beispieldaten/sudoku0.txt` bis `sudoku4.txt`)

Jede Datei enthält zwei Sudokus mit je 9 Zeilen aus 9 Zahlen (0 = leeres Feld), getrennt durch eine Leerzeile. `sudoku0.txt`
beginnt mit einem Byte-Order-Mark (beim Einlesen `encoding='utf-8-sig'` verwenden).

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Vollständige Suche (Brute Force) mit Abkürzungen:** Alle Umformungen durchprobieren ist möglich, aber teuer. Mit einer
  einfachen Vorprüfung (Anzahl der Zahlen je Zeile und Spalte) fallen fast alle Möglichkeiten sofort weg.
- **Invarianten:** Eine Eigenschaft, die sich durch die erlaubten Umformungen nicht ändert, hilft beim Aussortieren. Hier: wie viele
  Felder einer Zeile gefüllt sind (Vertauschen von Spalten und Umbenennen ändern das nicht).
- **Reihenfolge der Umformungen festlegen:** Man muss nicht jede Reihenfolge testen, wenn man zeigen kann, dass eine feste
  Reihenfolge schon alle Varianten erreicht.
- **Zuordnung prüfen:** Eine Umbenennung ist eine Abbildung von Ziffern auf Ziffern, die sich nie widersprechen darf und nie zwei
  Ziffern auf dieselbe abbildet (ein Dictionary und eine Menge reichen).

**Nützliche Funktionen und Bibliotheken**
- `itertools.permutations`: alle Umordnungen einer Liste, hier `permutations(range(3))` mit 6 Möglichkeiten
- `dict` und `set` für die Umbenennung
- Verschachtelte Listenschreibweise `[[... for j in ...] for i in ...]` zum Bauen und Drehen von Tabellen

## Die Idee

**Wie viele Umformungen gibt es?** Die drei Zeilenblöcke kann man auf 3! = 6 Arten umordnen, die drei Zeilen jedes Blocks ebenfalls
auf 6 Arten. Das gibt 6 · 6 · 6 · 6 = 1296 Anordnungen der Zeilen. Genauso 1296 für die Spalten. Dazu zwei Möglichkeiten (gedreht
oder nicht): 1296 · 1296 · 2 = **3 359 232** Kombinationen. Für jede müssten wir noch prüfen, ob eine passende Umbenennung der
Ziffern existiert. Das ginge, wäre in Python aber langsam.

**Die Reihenfolge.** Die Umformungen darf man in beliebiger Reihenfolge anwenden. Wir legen fest:

1. höchstens einmal um 90 Grad drehen,
2. Zeilen umordnen (Zeilenblöcke und Zeilen innerhalb der Blöcke),
3. Spalten umordnen,
4. Ziffern umbenennen.

Das verliert keine Varianten:

- Eine Umordnung der Zeilen ändert die Spaltenreihenfolge nicht und umgekehrt, ihre Reihenfolge ist egal. Das Umbenennen der
  Ziffern ändert nur die Zahlen, nicht ihre Orte, und lässt sich daher immer ans Ende schieben.
- Zeilenblöcke und Zeilen innerhalb der Blöcke lassen sich zu **einer** Umordnung der 9 Zeilen zusammenfassen, in der jeder Block
  zusammenbleibt. Mehrere solche Umordnungen hintereinander ergeben wieder eine.
- Eine Drehung um 180 Grad kehrt Zeilen- und Spaltenreihenfolge um, das ist eine Umordnung der Zeilen und der Spalten. Eine
  Drehung um 270 Grad ist 90 Grad plus 180 Grad. Also reichen „nicht drehen“ und „einmal um 90 Grad drehen“.

**Abkürzung.** Vertauscht man Zeilen, wandert jede Zeile samt ihren Zahlen an einen neuen Platz. Eine Zeile mit 4 gefüllten Feldern
kann also nur auf eine Zeile mit 4 gefüllten Feldern des Zielsudokus kommen (auch das Umbenennen ändert diese Zahl nicht). Die
Zeilenumordnung und die Spaltenumordnung kann man darum **unabhängig voneinander** vorab filtern: Nur Umordnungen, bei denen die
Anzahlen zusammenpassen, kommen in Frage. Erst die übrig gebliebenen Paare (Zeilen, Spalten) testen wir mit der Umbenennung. Das
ist die Idee der Optimierung in den Lösungshinweisen (dort auf Blockebene).

## Schritt 1: Einlesen

Ein Sudoku speichern wir als Liste von 9 Zeilen mit je 9 Zahlen. Die Datei enthält zwei Sudokus, getrennt durch eine Leerzeile.

In [ ]:
def lies_sudokus(dateiname):
    with open(dateiname, encoding='utf-8-sig') as f:
        text = f.read()
    sudokus = []
    for teil in text.strip().split('\n\n'):
        sudoku = []
        for zeile in teil.strip().splitlines():
            sudoku.append([int(z) for z in zeile.split()])
        sudokus.append(sudoku)
    return sudokus


start, ziel = lies_sudokus('beispieldaten/sudoku1.txt')
for zeile in start:
    print(*zeile)

0 0 0 2 0 0 0 0 0
0 1 0 0 6 0 9 2 0
0 0 0 0 0 1 5 0 0
8 9 0 6 0 2 0 0 1
0 0 5 0 0 0 0 0 4
0 0 4 0 7 0 0 6 0
0 0 0 0 0 8 3 0 0
5 0 7 4 0 0 0 0 0
0 0 0 0 1 0 0 0 0


## Schritt 2: Um 90 Grad im Uhrzeigersinn drehen

Beim Drehen im Uhrzeigersinn wird die **linke Spalte** (von unten nach oben gelesen) zur **obersten Zeile**. Allgemein: Das Feld in
Zeile `i`, Spalte `j` des gedrehten Sudokus ist das Feld in Zeile `8 - j`, Spalte `i` des alten.

In [ ]:
def drehe(sudoku):
    return [[sudoku[8 - j][i] for j in range(9)] for i in range(9)]


for zeile in drehe(start):
    print(*zeile)
print(drehe(drehe(drehe(drehe(start)))) == start)        # viermal drehen = wie vorher

0 5 0 0 0 8 0 0 0
0 0 0 0 0 9 0 1 0
0 7 0 4 5 0 0 0 0
0 4 0 0 0 6 0 0 2
1 0 0 7 0 0 0 6 0
0 0 8 0 0 2 1 0 0
0 0 3 0 0 0 5 9 0
0 0 0 6 0 0 0 2 0
0 0 0 0 4 1 0 0 0
True


## Schritt 3: Prüfen, ob es eine Umbenennung gibt

Zwei Sudokus `s` und `t` sind bis auf Umbenennung gleich, wenn es eine Zuordnung `f` gibt mit `t = f(s)` für jedes Feld. Wir gehen
alle Felder durch:

- Wurde die Ziffer aus `s` schon einmal zugeordnet, muss die frühere Zuordnung zur Ziffer aus `t` passen, sonst gibt es keine
  Umbenennung.
- Sonst legen wir die Zuordnung neu an. Hat aber schon eine andere Ziffer dieselbe Ziffer aus `t` bekommen, wäre die Umbenennung
  nicht umkehrbar und wir geben auf.

Das leere Feld `0` soll leer bleiben: Wir setzen `f[0] = 0` und markieren die `0` als schon vergeben, bevor wir anfangen.

In [ ]:
def finde_umbenennung(s, t):
    """Gibt ein Dictionary {Ziffer in s: Ziffer in t} zurück oder None, wenn es keine Umbenennung gibt."""
    f = {0: 0}                  # leeres Feld bleibt leer
    benutzt = {0}               # Ziffern, die in t schon vergeben sind
    for i in range(9):
        for j in range(9):
            a = s[i][j]
            b = t[i][j]
            if a in f:
                if f[a] != b:
                    return None
            else:
                if b in benutzt:
                    return None
                f[a] = b
                benutzt.add(b)
    return f


print(finde_umbenennung(start, start))
print(finde_umbenennung(drehe(start), ziel))

{0: 0, 2: 2, 1: 1, 6: 6, 9: 9, 5: 5, 8: 8, 4: 4, 7: 7, 3: 3}
None


## Schritt 4: Alle sinnvollen Umordnungen der Zeilen

Eine Umordnung beschreiben wir als Liste `m` mit 9 Einträgen: **An Platz `i` des neuen Sudokus steht die Zeile `m[i]` des alten.**
Dabei muss jeder Block zusammenbleiben. Wir bauen alle 1296 Listen so: Die Reihenfolge der Blöcke ist `bloecke` (eine Umordnung
von 0, 1, 2), und innerhalb jedes Blocks gibt es noch eine Umordnung `innen[b]`. Neuer Platz `3*b + k` bekommt die alte Zeile
`3 * bloecke[b] + innen[b][k]`. Für Spalten gilt dasselbe, deshalb sind es dieselben 1296 Listen.

Jetzt kommt die **Abkürzung**: `anzahl_alt[z]` ist die Zahl gefüllter Felder in Zeile `z` des Quellsudokus, `anzahl_neu[i]` im
Zielsudoku. Wir behalten nur die `m` mit `anzahl_alt[m[i]] == anzahl_neu[i]` für alle `i`. Für Spalten nehmen wir dieselbe Funktion
mit den Anzahlen je Spalte.

In [ ]:
from itertools import permutations

UMORDNUNGEN = list(permutations(range(3)))          # die 6 Umordnungen von 0, 1, 2


def alle_umordnungen():
    ergebnis = []
    for bloecke in UMORDNUNGEN:
        for innen0 in UMORDNUNGEN:
            for innen1 in UMORDNUNGEN:
                for innen2 in UMORDNUNGEN:
                    innen = [innen0, innen1, innen2]
                    m = []
                    for b in range(3):
                        for k in range(3):
                            m.append(3 * bloecke[b] + innen[b][k])
                    ergebnis.append(m)
    return ergebnis


ALLE = alle_umordnungen()
print(len(ALLE), ALLE[0], ALLE[-1])


def zeilen_anzahlen(sudoku):
    return [sum(1 for z in zeile if z != 0) for zeile in sudoku]


def spalten_anzahlen(sudoku):
    return [sum(1 for i in range(9) if sudoku[i][j] != 0) for j in range(9)]


def passende_umordnungen(anzahl_alt, anzahl_neu):
    ergebnis = []
    for m in ALLE:
        passt = True
        for i in range(9):
            if anzahl_alt[m[i]] != anzahl_neu[i]:
                passt = False
                break
        if passt:
            ergebnis.append(m)
    return ergebnis


zeilen_m = passende_umordnungen(zeilen_anzahlen(drehe(start)), zeilen_anzahlen(ziel))
spalten_m = passende_umordnungen(spalten_anzahlen(drehe(start)), spalten_anzahlen(ziel))
print('passende Zeilenumordnungen:', len(zeilen_m), ' passende Spaltenumordnungen:', len(spalten_m))

1296 [0, 1, 2, 3, 4, 5, 6, 7, 8] [8, 7, 6, 5, 4, 3, 2, 1, 0]
passende Zeilenumordnungen: 48  passende Spaltenumordnungen: 1


Von je 1296 Möglichkeiten bleiben also nur wenige übrig. Statt 3 359 232 Kombinationen sind nur noch die Paare aus diesen beiden
Listen zu prüfen (und das für „nicht gedreht“ und „gedreht“).

## Schritt 5: Die Funktion `ist_variante`

Für beide Möglichkeiten (nicht drehen, drehen) bilden wir das Quellsudoku `q` und bestimmen die passenden Zeilen- und
Spaltenumordnungen. Für jedes Paar `(zm, sm)` bauen wir `neu[i][j] = q[zm[i]][sm[j]]` und prüfen die Umbenennung. Wir geben die
Beschreibung der Umformung zurück, oder `None`. Zusätzlich zählen wir, wie viele Kombinationen geprüft wurden.

In [ ]:
def ist_variante(s, t):
    """Gibt (gedreht, zeilen, spalten, umbenennung, geprueft) zurück oder None, wenn t keine Variante von s ist."""
    geprueft = 0
    for gedreht in [False, True]:
        q = drehe(s) if gedreht else s
        zeilen_liste = passende_umordnungen(zeilen_anzahlen(q), zeilen_anzahlen(t))
        spalten_liste = passende_umordnungen(spalten_anzahlen(q), spalten_anzahlen(t))
        for zm in zeilen_liste:
            for sm in spalten_liste:
                neu = [[q[zm[i]][sm[j]] for j in range(9)] for i in range(9)]
                geprueft += 1
                f = finde_umbenennung(neu, t)
                if f is not None:
                    return gedreht, zm, sm, f, geprueft
    return None

## Schritt 6: Umformungen lesbar ausgeben

Aus `zm` (Zeilen) und `sm` (Spalten) wird eine Beschreibung in Worten:

- Die Reihenfolge der **Blöcke**: Der neue Block `b` ist der alte Block `m[3*b] // 3`.
- Die Reihenfolge **innerhalb der Blöcke**: Die neue Position `k` im Block `b` ist die alte Position `m[3*b + k] % 3`.
- Die **Umbenennung**: für jede Ziffer 1 bis 9 die neue.

Alles zählen wir ab 1 und schreiben jeweils die alten Nummern in der Reihenfolge der neuen Plätze auf. Die Funktion `wende_an`
wendet die Umformung auf das Startsudoku an. Damit prüfen wir das Ergebnis unabhängig vom Suchen.

In [ ]:
def beschreibe(ergebnis):
    gedreht, zm, sm, f, geprueft = ergebnis
    zeilen = []
    if gedreht:
        zeilen.append('Um 90 Grad im Uhrzeigersinn drehen')
    else:
        zeilen.append('Nicht drehen')
    for name, m in [('Zeilen', zm), ('Spalten', sm)]:
        bloecke = [m[3 * b] // 3 + 1 for b in range(3)]
        zeilen.append(name + 'blöcke (alte Nummern auf den neuen Plätzen): ' + ' '.join(str(b) for b in bloecke))
        for b in range(3):
            innen = [m[3 * b + k] % 3 + 1 for k in range(3)]
            zeilen.append('  neuer ' + name + 'block ' + str(b + 1) + ': alte Positionen ' + ' '.join(str(p) for p in innen))
    zeilen.append('Umbenennung: ' + ', '.join(str(a) + ' -> ' + str(f[a]) for a in range(1, 10)))
    return '\n'.join(zeilen)


def wende_an(s, ergebnis):
    gedreht, zm, sm, f, geprueft = ergebnis
    q = drehe(s) if gedreht else s
    return [[f[q[zm[i]][sm[j]]] for j in range(9)] for i in range(9)]


ergebnis = ist_variante(start, ziel)
print(beschreibe(ergebnis))
print('Stimmt:', wende_an(start, ergebnis) == ziel)

Um 90 Grad im Uhrzeigersinn drehen
Zeilenblöcke (alte Nummern auf den neuen Plätzen): 2 3 1
  neuer Zeilenblock 1: alte Positionen 1 2 3
  neuer Zeilenblock 2: alte Positionen 1 2 3
  neuer Zeilenblock 3: alte Positionen 1 2 3
Spaltenblöcke (alte Nummern auf den neuen Plätzen): 2 3 1
  neuer Spaltenblock 1: alte Positionen 1 2 3
  neuer Spaltenblock 2: alte Positionen 1 2 3
  neuer Spaltenblock 3: alte Positionen 1 2 3
Umbenennung: 1 -> 1, 2 -> 2, 3 -> 3, 4 -> 4, 5 -> 5, 6 -> 6, 7 -> 7, 8 -> 8, 9 -> 9
Stimmt: True


In `sudoku1.txt` wird gedreht und es werden Blöcke vertauscht; die Ziffern bleiben, wie sie sind. Das passt zu den Lösungshinweisen
(Rotation, Zeilen- und Spaltenblöcke vertauscht). Die Lösung ist nicht eindeutig: Jede Umformung, die zum Ziel führt, ist richtig.

## Schritt 7: Alle Beispiele

Wir prüfen zu jedem Ergebnis die Umformung mit `wende_an`, damit wir sicher sind, dass sie wirklich das Zielsudoku erzeugt. Dazu
zeigen wir die Zahl der geprüften Kombinationen (statt Laufzeiten).

In [ ]:
def bearbeite(nr):
    s, t = lies_sudokus('beispieldaten/sudoku' + str(nr) + '.txt')
    ergebnis = ist_variante(s, t)
    print('=== sudoku' + str(nr) + '.txt ===')
    if ergebnis is None:
        print('Keine Variante.')
    else:
        print(beschreibe(ergebnis))
        print('geprüfte Kombinationen:', ergebnis[4], '  Probe:', wende_an(s, ergebnis) == t)
    print()
    return ergebnis


ERGEBNISSE = [bearbeite(nr) for nr in range(5)]

=== sudoku0.txt ===
Nicht drehen
Zeilenblöcke (alte Nummern auf den neuen Plätzen): 1 2 3
  neuer Zeilenblock 1: alte Positionen 2 1 3
  neuer Zeilenblock 2: alte Positionen 1 2 3
  neuer Zeilenblock 3: alte Positionen 1 3 2
Spaltenblöcke (alte Nummern auf den neuen Plätzen): 1 2 3
  neuer Spaltenblock 1: alte Positionen 3 1 2
  neuer Spaltenblock 2: alte Positionen 2 3 1
  neuer Spaltenblock 3: alte Positionen 3 2 1
Umbenennung: 1 -> 1, 2 -> 2, 3 -> 3, 4 -> 4, 5 -> 5, 6 -> 6, 7 -> 7, 8 -> 8, 9 -> 9
geprüfte Kombinationen: 34   Probe: True

=== sudoku1.txt ===
Um 90 Grad im Uhrzeigersinn drehen
Zeilenblöcke (alte Nummern auf den neuen Plätzen): 2 3 1
  neuer Zeilenblock 1: alte Positionen 1 2 3
  neuer Zeilenblock 2: alte Positionen 1 2 3
  neuer Zeilenblock 3: alte Positionen 1 2 3
Spaltenblöcke (alte Nummern auf den neuen Plätzen): 2 3 1
  neuer Spaltenblock 1: alte Positionen 1 2 3
  neuer Spaltenblock 2: alte Positionen 1 2 3
  neuer Spaltenblock 3: alte Positionen 1 2 3
Umbenennun

Auswertung:

- `sudoku0.txt`: Nur einzelne Zeilen und Spalten sind vertauscht, die Ziffern sind gleich geblieben, es wird nicht gedreht.
- `sudoku1.txt`: Drehung und Vertauschung von Blöcken.
- `sudoku2.txt`: Das Beispiel der Aufgabe: Spaltenblöcke 1 und 3 vertauscht, Zeilen 5 und 6 vertauscht, Ziffern um 1 erhöht
  (9 → 1). Die Umbenennung stimmt mit den Lösungshinweisen überein.
- `sudoku3.txt`: **Keine Variante.** Schon die Vorprüfung scheitert: Weder gedreht noch ungedreht gibt es eine Umordnung
  der Zeilen oder der Spalten, bei der die Anzahlen gefüllter Felder zum Zielsudoku passen.
- `sudoku4.txt`: Alles kommt vor: Drehung, Blöcke, Zeilen, Spalten und eine Umbenennung.

Unsere Beschreibungen können von denen der Lösungshinweise abweichen (andere Reihenfolge, gleichwertige Umformung). Entscheidend ist,
dass die Probe `wende_an(...) == Ziel` stimmt.

## Noch ein Test mit selbst durchgemischten Sudokus

Damit wir nicht nur die fünf Beispiele kennen, mischen wir ein Sudoku zufällig durch (Drehen, Zeilen, Spalten und Ziffern
umbenennen) und prüfen, dass `ist_variante` die Variante immer findet. Dann ändern wir im durchgemischten Sudoku eine Ziffer und
prüfen, dass es keine Variante mehr ist.

In [ ]:
import random


def mische(s):
    q = s
    if random.random() < 0.5:
        q = drehe(q)
    zm = random.choice(ALLE)
    sm = random.choice(ALLE)
    ziffern = list(range(1, 10))
    random.shuffle(ziffern)
    f = {0: 0}
    for a in range(1, 10):
        f[a] = ziffern[a - 1]
    return [[f[q[zm[i]][sm[j]]] for j in range(9)] for i in range(9)]


random.seed(5)
basis, _ = lies_sudokus('beispieldaten/sudoku4.txt')
gefunden = 0
for versuch in range(20):
    gemischt = mische(basis)
    e = ist_variante(basis, gemischt)
    if e is not None and wende_an(basis, e) == gemischt:
        gefunden += 1
print('Varianten gefunden:', gefunden, 'von 20')

# ein gefülltes Feld durch eine andere Ziffer ersetzen
gemischt = mische(basis)
stelle = [(i, j) for i in range(9) for j in range(9) if gemischt[i][j] != 0][0]
gemischt[stelle[0]][stelle[1]] = gemischt[stelle[0]][stelle[1]] % 9 + 1
print('nach der Änderung eine Variante:', ist_variante(basis, gemischt) is not None)

Varianten gefunden: 20 von 20
nach der Änderung eine Variante: False


## Zum Schluss: eine Lösung mit MiniZinc

Hier suchen wir keine Zahl und keinen Weg, sondern **Umformungen, die zusammenpassen müssen**: Das ist ein **Constraint-Problem**.
Statt Schleifen über alle Umordnungen beschreiben wir mit Variablen, was gelten soll, und überlassen die Suche dem Solver.

**Variablen:**
- `gedreht` (0 oder 1): gedreht oder nicht.
- `zblock[b]`: welcher alte Zeilenblock im neuen Block `b` steht (alle verschieden: `alldifferent`). Entsprechend `sblock`.
- `zinnen[b, k]`: welche alte Position innerhalb des Blocks im neuen Block `b` an der Stelle `k` steht (in jedem Block alle
  verschieden). Entsprechend `sinnen`.
- `f[0..9]`: die Umbenennung, ebenfalls mit `alldifferent`, und `f[0] = 0`, weil leere Felder leer bleiben.

**Bedingung:** Für jedes Feld `(i, j)` des Ziels gilt `t[i, j] = f[altes Feld]`. Das alte Feld wird mit den Variablen **als Index**
gesucht: `s[gedreht, 3 * zblock[i div 3] + zinnen[...], ...]`. Damit die Drehung eine Variable sein kann, legen wir in der
Datendatei beide Fassungen des Startsudokus ab (`s[0]` ungedreht, `s[1]` gedreht). In MiniZinc zählen wir ab 0.

Die MiniZinc-Blöcke unten sind **kein Python**: Kopiere sie in die MiniZinc IDE (oder in den
[MiniZinc Playground](https://play.minizinc.org/)) und führe sie dort aus. Die Datendatei (`.dzn`) erzeugt die folgende Python-Zelle
aus den Beispieldaten.

In [ ]:
def als_dzn(s, t):
    werte_s = [z for zeile in s for z in zeile] + [z for zeile in drehe(s) for z in zeile]
    werte_t = [z for zeile in t for z in zeile]
    return ('s = array3d(0..1, 0..8, 0..8, ' + str(werte_s) + ');\n'
            't = array2d(0..8, 0..8, ' + str(werte_t) + ');')


s2, t2 = lies_sudokus('beispieldaten/sudoku2.txt')
print(als_dzn(s2, t2)[:100] + ' ...')

s = array3d(0..1, 0..8, 0..8, [5, 0, 0, 0, 8, 0, 0, 4, 9, 0, 0, 0, 5, 0, 0, 0, 3, 0, 0, 6, 7, 3, 0,  ...


```minizinc
include "alldifferent.mzn";

array[0..1, 0..8, 0..8] of int: s;      % s[0] = Start, s[1] = Start um 90 Grad gedreht
array[0..8, 0..8] of int: t;            % Ziel

var 0..1: gedreht;
array[0..2] of var 0..2: zblock;        % neuer Zeilenblock b ist der alte Zeilenblock zblock[b]
array[0..2] of var 0..2: sblock;
array[0..2, 0..2] of var 0..2: zinnen;  % neue Zeile k im Block b ist die alte Position zinnen[b, k]
array[0..2, 0..2] of var 0..2: sinnen;
array[0..9] of var 0..9: f;             % Umbenennung, f[0] = 0

constraint alldifferent(zblock) /\ alldifferent(sblock) /\ alldifferent(f);
constraint forall(b in 0..2)(alldifferent([zinnen[b, k] | k in 0..2]) /\ alldifferent([sinnen[b, k] | k in 0..2]));
constraint f[0] = 0;

constraint forall(i, j in 0..8)(
  t[i, j] = f[ s[gedreht, 3 * zblock[i div 3] + zinnen[i div 3, i mod 3],
                          3 * sblock[j div 3] + sinnen[j div 3, j mod 3]] ]);

solve satisfy;

output ["gedreht = \(gedreht)\nZeilenblöcke = \(zblock)\nSpaltenblöcke = \(sblock)\n",
        "Zeilen innen = \(zinnen)\nSpalten innen = \(sinnen)\nUmbenennung = \(f)\n"];
```

Ergebnis für `sudoku2.txt` (Gecode):

**Ausgabe:**

```
gedreht = 0
Zeilenblöcke = [0, 1, 2]
Spaltenblöcke = [2, 1, 0]
Zeilen innen = [0, 1, 2, 0, 2, 1, 0, 1, 2]
Spalten innen = [0, 1, 2, 0, 1, 2, 0, 1, 2]
Umbenennung = [0, 2, 3, 4, 5, 6, 7, 8, 9, 1]
----------
```

Das ist eine zulässige Umformung, die von der Python-Ausgabe abweichen kann (es gibt oft mehrere). Bei `sudoku3.txt` meldet der Solver,
dass es keine Lösung gibt:

**Ausgabe:**

```
=====UNSATISFIABLE=====
```

Alle fünf Beispiele rechnet der Solver in Bruchteilen einer Sekunde (gemessen mit Gecode und Chuffed). Die Vorfilterung nach der Zahl
der Einträge brauchen wir hier nicht selbst zu programmieren: Der Solver streicht ungeeignete Werte von selbst, sobald ein Feld nicht
passt.